# Natural Language → SQL Pipeline (LangChain + Llama 3.1 + MySQL)

Implements the flow:

`User Query → Prompt Template (schema/rules) → Llama 3.1 → SQL Validator → SQLAlchemy → MySQL → Results → LLM (English answer) → User Response`

Fill in the connection details in the **Config** cell, then run all cells.

In [2]:
# 2. Config

# --- MySQL connection ---
DB_USER = "sql_agent"
DB_PASSWORD = "Delhi@369"
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "sakila"

# --- LLM: Llama 3.1 ---
# Choose ONE provider below.
#   A) Groq-hosted Llama 3.1 (fast, needs GROQ_API_KEY)
#   B) Local Llama 3.1 via Ollama (needs `ollama pull llama3.1` running locally)
LLM_PROVIDER = "ollama"   # "groq" or "ollama"

import os
os.environ["GROQ_API_KEY"] = "your_groq_api_key"   # only needed if LLM_PROVIDER == "groq"

# Tables the LLM is allowed to query (used by both the prompt and the validator)
# Sakila sample database (DVD rental store)
ALLOWED_TABLES = {
    "film", "film_category", "category", "film_actor", "actor",
    "inventory", "rental", "payment", "customer", "store", "staff",
    "address", "city", "country"
}


## 3. Database connection (SQLAlchemy)

In [3]:
from urllib.parse import quote_plus
from sqlalchemy import create_engine, inspect, text

DB_USER = "sql_agent"
DB_PASSWORD = quote_plus("Delhi@369")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "sakila"

DATABASE_URL = (
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)


def get_schema_info(engine, allowed_tables):
    inspector = inspect(engine)
    schema_lines = []

    for table in inspector.get_table_names():
        if table not in allowed_tables:
            continue

        cols = inspector.get_columns(table)
        col_desc = ", ".join(
            f"{c['name']} ({c['type']})" for c in cols
        )
        schema_lines.append(f"Table `{table}`: {col_desc}")

        for fk in inspector.get_foreign_keys(table):
            schema_lines.append(
                f"FK: {table}.{fk['constrained_columns']} -> "
                f"{fk['referred_table']}.{fk['referred_columns']}"
            )

    return "\n".join(schema_lines)


try:
    with engine.connect() as conn:
        print("✅ Connected to MySQL")
        print(conn.execute(text("SELECT DATABASE();")).fetchone())

    schema_info = get_schema_info(engine, ALLOWED_TABLES)
    print(schema_info)

except Exception as e:
    print(e)

✅ Connected to MySQL
('sakila',)
Table `actor`: actor_id (SMALLINT), first_name (VARCHAR(45)), last_name (VARCHAR(45)), last_update (TIMESTAMP)
Table `address`: address_id (SMALLINT), address (VARCHAR(50)), address2 (VARCHAR(50)), district (VARCHAR(20)), city_id (SMALLINT), postal_code (VARCHAR(10)), phone (VARCHAR(20)), location (NULL), last_update (TIMESTAMP)
FK: address.['city_id'] -> city.['city_id']
Table `category`: category_id (TINYINT), name (VARCHAR(25)), last_update (TIMESTAMP)
Table `city`: city_id (SMALLINT), city (VARCHAR(50)), country_id (SMALLINT), last_update (TIMESTAMP)
FK: city.['country_id'] -> country.['country_id']
Table `country`: country_id (SMALLINT), country (VARCHAR(50)), last_update (TIMESTAMP)
Table `customer`: customer_id (SMALLINT), store_id (TINYINT), first_name (VARCHAR(45)), last_name (VARCHAR(45)), email (VARCHAR(50)), address_id (SMALLINT), active (TINYINT), create_date (DATETIME), last_update (TIMESTAMP)
FK: customer.['address_id'] -> address.['addre

/var/folders/hf/hb787sb57hn03tn2vy4t03hw0000gn/T/ipykernel_10799/2605950620.py:25: SAWarning: Did not recognize type 'geometry' of column 'location'
  cols = inspector.get_columns(table)


## 4. Prompt template (LangChain)

In [4]:
from langchain_core.prompts import ChatPromptTemplate

SQL_RULES = """
Rules:
- Only use tables from the schema provided below.
- Only generate SELECT statements. Never write INSERT, UPDATE, DELETE, DROP, ALTER, or TRUNCATE.
- Always use explicit JOINs based on the foreign key relationships given.
- Return ONLY the raw SQL query. No explanation, no markdown code fences.
- If the question cannot be answered with the given schema, return exactly: CANNOT_ANSWER
"""

sql_prompt = ChatPromptTemplate.from_template(
    """You are a MySQL expert. Convert the user's question into a single SQL query.

Schema:
{schema_info}

{sql_rules}

User question: {user_query}

SQL query:"""
)


/Users/maddukuri/anaconda3/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


## 5. LLM (Llama 3.1)

In [5]:
if LLM_PROVIDER == "groq":
    from langchain_groq import ChatGroq
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
elif LLM_PROVIDER == "ollama":
    from langchain_community.chat_models import ChatOllama
    llm = ChatOllama(model="qwen3:8b", temperature=0)
else:
    raise ValueError("LLM_PROVIDER must be 'groq' or 'ollama'")

sql_chain = sql_prompt | llm

def generate_sql(user_query: str, schema_info: str) -> str:
    response = sql_chain.invoke({
        "schema_info": schema_info,
        "sql_rules": SQL_RULES,
        "user_query": user_query,
    })
    sql = response.content.strip()
    # Strip accidental markdown fences
    sql = sql.replace("```sql", "").replace("```", "").strip()
    return sql


/var/folders/hf/hb787sb57hn03tn2vy4t03hw0000gn/T/ipykernel_10799/1733946677.py:6: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="qwen3:8b", temperature=0)


## 6. SQL Validator Layer

In [6]:
import re
import sqlparse

FORBIDDEN_KEYWORDS = {
    "DROP", "DELETE", "UPDATE", "INSERT", "ALTER",
    "TRUNCATE", "CREATE", "GRANT", "REVOKE", "EXEC", "EXECUTE",
}

class SQLValidationError(Exception):
    pass

def validate_sql(sql: str, allowed_tables: set) -> str:
    if not sql or sql.strip().upper() == "CANNOT_ANSWER":
        raise SQLValidationError("Model could not translate the question into SQL.")

    # 1. Syntax check
    parsed = sqlparse.parse(sql)
    if not parsed or not parsed[0].tokens:
        raise SQLValidationError("Could not parse SQL — invalid syntax.")

    statement = parsed[0]
    stmt_type = statement.get_type()

    # 2. SELECT-only check
    if stmt_type != "SELECT":
        raise SQLValidationError(f"Only SELECT statements are allowed, got: {stmt_type}")

    # 3. Forbidden keyword check (defense in depth beyond stmt_type)
    tokens_upper = sql.upper()
    for kw in FORBIDDEN_KEYWORDS:
        if re.search(rf"\b{kw}\b", tokens_upper):
            raise SQLValidationError(f"Forbidden keyword detected: {kw}")

    # 4. Allowed-tables check
    # crude but effective: pull identifiers after FROM / JOIN
    referenced_tables = set(re.findall(r"(?:FROM|JOIN)\s+`?(\w+)`?", sql, re.IGNORECASE))
    disallowed = referenced_tables - allowed_tables
    if disallowed:
        raise SQLValidationError(f"Query references disallowed table(s): {disallowed}")

    # 5. Single statement only (no stacked queries)
    if len(sqlparse.split(sql)) > 1:
        raise SQLValidationError("Multiple statements are not allowed.")

    return sql


## 7. Execute query (SQLAlchemy → MySQL)

In [7]:
import pandas as pd

def run_query(engine, sql: str) -> pd.DataFrame:
    with engine.connect() as conn:
        result = conn.execute(text(sql))
        rows = result.fetchall()
        columns = result.keys()
    return pd.DataFrame(rows, columns=columns)


## 8. LLM converts results into an English answer

In [8]:
answer_prompt = ChatPromptTemplate.from_template(
    """The user asked: "{user_query}"

The SQL query below was run and returned this data (as a table):
{query_result}

Write a concise, natural-language answer to the user's question based only on this data.
Do not mention SQL or databases in your answer."""
)

answer_chain = answer_prompt | llm

def explain_results(user_query: str, df: pd.DataFrame) -> str:
    result_str = df.to_string(index=False) if not df.empty else "No rows returned."
    response = answer_chain.invoke({
        "user_query": user_query,
        "query_result": result_str,
    })
    return response.content.strip()


## 9. Full pipeline

In [9]:
def ask_database(user_query: str, retries: int = 1):
    """
    Full pipeline: NL question -> SQL -> validate -> execute -> NL answer.
    Retries once with the LLM if validation or execution fails.
    """
    last_error = None
    for attempt in range(retries + 1):
        try:
            sql = generate_sql(user_query, schema_info)
            sql = validate_sql(sql, ALLOWED_TABLES)
            df = run_query(engine, sql)
            answer = explain_results(user_query, df)
            return {
                "question": user_query,
                "sql": sql,
                "result": df,
                "answer": answer,
            }
        except (SQLValidationError, Exception) as e:
            last_error = e
            continue
    return {
        "question": user_query,
        "sql": None,
        "result": None,
        "answer": f"Sorry, I couldn't answer that. ({last_error})",
    }


## 10. Example usage

In [10]:
result = ask_database("What were the top 5 most rented films?")

print("Generated SQL:\n", result["sql"])
print("\nAnswer:\n", result["answer"])
result["result"]


Generated SQL:
 SELECT f.film_id, f.title, COUNT(r.rental_id) AS rental_count
FROM rental r
JOIN inventory i ON r.inventory_id = i.inventory_id
JOIN film f ON i.film_id = f.film_id
GROUP BY f.film_id
ORDER BY rental_count DESC
LIMIT 5;

Answer:
 The top 5 most rented films are:  
1. **BUCKET BROTHERHOOD** with 34 rentals  
2. **ROCKETEER MOTHER** with 33 rentals  
3. **FORWARD TEMPLE** with 32 rentals  
4. **GRIT CLOCKWORK** with 32 rentals  
5. **JUGGLER HARDLY** with 32 rentals  

Note that the last three films tied for third place.


,film_id,title,rental_count
0,103,BUCKET BROTHERHOOD,34
1,738,ROCKETEER MOTHER,33
2,331,FORWARD TEMPLE,32
3,382,GRIT CLOCKWORK,32
4,489,JUGGLER HARDLY,32
